In [ ]:
%pip install yfinance torchmetrics

In [ ]:
import yfinance as yf
import matplotlib.pyplot as plt
import pandas as pd
import torch
import numpy as np
from torch import nn

In [ ]:
# feel free to play around with different ticker values
ticker = "NVDA"
description = "Nvidia"

# get the data for the last 10 years. This is daily data by default, so it's not actually that much.
data = yf.download(ticker, period="10y")

import matplotlib.pyplot as plt

plt.plot(data['Close'])
plt.ylabel(f'{description} Closing Price (USD)')
plt.xticks(rotation=45)
plt.show()

print(data.info())
data.head()

In [ ]:
train_end = '2023-12-31'
val_end = '2024-12-31'
scale_max = 150

train_np = data['Close'][:train_end].values / scale_max
val_np = data['Close'][train_end:val_end].values / scale_max
test_np = data['Close'][val_end:].values / scale_max

## Helper functions
The following cell contains helper functions that can be used as-is.

In [ ]:
def get_device():
    if torch.cuda.is_available():
        device = "cuda"
    elif torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"

    return device


def evaluate_tm(model, data_loader, metric, device, return_series=False):
    model.eval()
    metric.reset()
    yhat = None
    if return_series:
        yhat = []
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
            if return_series:
                yhat.append(y_pred.detach().numpy())
    
    if return_series:
        return metric.compute(), np.concat(yhat)
    else:
        return metric.compute()


def train(
    model,
    optimizer,
    loss_fn,
    metric,
    train_loader,
    valid_loader,
    n_epochs,
    device,
    save_path=None,
):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric, device).item()
        )
        print(
            f"Epoch {epoch + 1}/{n_epochs}, "
            f"train loss: {history['train_losses'][-1]:.4f}, "
            f"train metric: {history['train_metrics'][-1]:.4f}, "
            f"valid metric: {history['valid_metrics'][-1]:.4f}"
        )
        if save_path:
            # save the weights every epoch
            torch.save(model.state_dict(), save_path)
    return history


## Define the data loaders
Adapted from the [Chapter 13 notebook](https://github.com/ageron/handson-mlp/blob/main/13_processing_sequences_using_rnns_and_cnns.ipynb) of the Hands-on ML book.

In [ ]:
import torch
import torch.nn as nn
import torchmetrics

class TimeSeriesDataset(torch.utils.data.Dataset):
    def __init__(self, series, window_length):
        # I moved the conversion to tensor in here, so it's expecting a numpy array
        self.series = torch.FloatTensor(series)
        self.window_length = window_length

    def __len__(self):
        return len(self.series) - self.window_length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        end = idx + self.window_length
        window = self.series[idx : end]
        target = self.series[end : end]
        return window, target

In [ ]:
# TODO: Define a window size for training and create TimeSeriesDatasets for train, val, test

In [ ]:
# create the data loaders that handle batching and shuffling
from torch.utils.data import DataLoader
torch.manual_seed(1234)
# TODO: create data loaders for each of the TimeSeriesDatasets, e.g.
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

In [ ]:
# Inspect the dimensions of the batches
batch = next(iter(train_loader))
print(batch[0].shape)
print(batch[1].shape)

In [ ]:
# Now define an RNN
class SimpleRnnModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.output = nn.LazyLinear(output_size)

    def forward(self, X: torch.FloatTensor):
        outputs, _ = self.rnn(X)
        return self.output(outputs[:,-1]).squeeze()

In [ ]:
# Note: hidden_size is the number of time steps going backwards that get a 
simple_rnn = SimpleRnnModel(1, 1, 1)
device = get_device()
n_epochs = 50
optimizer = torch.optim.Adam(simple_rnn.parameters())
loss = nn.HuberLoss()
metric = torchmetrics.MeanAbsoluteError()
history = train(
    simple_rnn,
    optimizer,
    loss,
    metric,
    train_loader,
    valid_loader,
    n_epochs,
    device,
)

In [ ]:
#TODO: pass return_series=True to evaluate_tm and plot predictions for the first validation sample